# Reproducing the SVI paper's introductory PCA example

This notebook implements the motivating pipe-mixing example from Camacho, Picó and Ferrer, *Data understanding with PCA: Structural and Variance Information plots* (2010). Three input liquids, $F_1$, $F_2$, and $F_3$, are mixed in two pipes. The PCA calibration matrix is $X=[F_1,F_2,F_{123}]$, with 100 observations and three variables.

$$F_{12}=\frac{4}{5}F_1+\frac{1}{5}F_2, \qquad F_{123}=\frac{16}{25}F_1+\frac{4}{25}F_2+\frac{5}{25}F_3.$$

The article specifies the generating equations, not a fixed data table. The values below are therefore a seeded simulation of the same 100-sample construction. Individual loading values and arbitrary PCA signs can differ from the printed figures, while the structural relationships remain the same.

## 1. Imports and reproducibility

Run this notebook from an environment where the local package is installed (for example, `pip install -e .` from the repository root).

In [1]:
import numpy as np
import pandas as pd

from pca_tools import PCA

RANDOM_SEED = 2010
N_OBSERVATIONS = 100
rng = np.random.default_rng(RANDOM_SEED)

## 2. Build the pipe-mixing data

$F_1$, $F_2$, and $F_3$ are independent normally distributed input flows with mean zero and standard deviation one. First, $F_1$ and $F_2$ form the intermediate flow $F_{12}$. Then $F_{12}$ and $F_3$ form the outlet flow $F_{123}$. As in the paper, only the two input flows $F_1$, $F_2$ and the final flow $F_{123}$ are used for PCA.

In [2]:
f1, f2, f3 = rng.normal(size=(3, N_OBSERVATIONS))

F12 = (4 / 5) * f1 + (1 / 5) * f2
F123 = (16 / 25) * f1 + (4 / 25) * f2 + (5 / 25) * f3

flows = pd.DataFrame(
    {
        "F1": f1,
        "F2": f2,
        "F3": f3,
        "F12": F12,
        "F123": F123,
    }
)

X = flows[["F1", "F2", "F123"]]

display(flows.head())
display(X.corr().round(3))

,F1,F2,F3,F12,F123
0,-0.771906,0.600455,-0.874782,-0.497434,-0.572903
1,-0.671288,0.921000,0.251772,-0.352831,-0.231910
2,0.527803,-1.271075,0.088649,0.168027,0.152151
3,-0.451545,0.118499,0.017354,-0.337536,-0.266558
4,1.813673,-0.195897,0.853580,1.411759,1.300123


,F1,F2,F123
F1,1.000,-0.009,0.934
F2,-0.009,1.000,0.242
F123,0.934,0.242,1.000


The weighting creates a very strong linear relationship between $F_1$ and $F_{123}$, a weaker relationship between $F_2$ and $F_{123}$, and no designed relationship between $F_1$ and $F_2$. This unequal correlation pattern is the reason that interpreting loading vectors alone can be misleading.

## 3. Fit PCA

The three retained variables are standardized, so this is a correlation-based PCA. All three components are retained to reproduce the paper's cumulative structural analysis; the first component dominates because of the strong $F_1$–$F_{123}$ relationship.

In [3]:
model = PCA(n_comps=3, standardize=True, alpha=0.99).fit(X)

explained_variance = pd.DataFrame(
    {
        "explained_variance_ratio": model._explained_variance,
        "cumulative_explained_variance": model.get_rsquared_acc(),
    },
    index=model._scores.columns,
)
explained_variance.round(3)

,explained_variance_ratio,cumulative_explained_variance
PC_1,0.654,0.654
PC_2,0.335,0.989
PC_3,0.011,1.000


## 4. Ordinary PCA interpretation

The loading bar plots show the contribution of each measured flow to the first two latent directions. The biplot places observations and loading vectors in the same PC1–PC2 view. Its signs may be reflected when PCA is re-run; relative positions and magnitudes are what matter.

In [4]:
model.loadings_barplot(1)

alt.Chart(...)

In [5]:
model.loadings_barplot(2)

alt.Chart(...)

In [6]:
model.biplot(1, 2)

alt.LayerChart(...)

## 5. Structural projection matrices from the paper

For the first $A$ components, the paper uses $Q_A=P_A P_A^\mathsf{T}$. Its diagonal, $\alpha_{i,A}$, is the **self-explanatory power** of variable $i$: the amount of that variable's direction represented by the selected PCA subspace. The off-diagonal terms describe pairwise structural relationships in that subspace.

The cells below reproduce these matrices for one, two, and three retained components. With all three components, $Q_3$ is the identity matrix (up to numerical precision), because this example has three variables and three retained PCs.

In [7]:
def projection_matrix(loadings: pd.DataFrame, n_components: int) -> pd.DataFrame:
    """Return Q_A = P_A P_A^T for the first A loading columns."""
    p_a = loadings.iloc[:, :n_components].to_numpy()
    return pd.DataFrame(
        p_a @ p_a.T,
        index=loadings.index,
        columns=loadings.index,
    )

for n_components in range(1, 4):
    print(f"Q_{n_components}")
    display(projection_matrix(model._loadings, n_components).round(3))

Q_1


,F1,F2,F123
F1,0.470,0.117,0.485
F2,0.117,0.029,0.121
F123,0.485,0.121,0.501


Q_2


,F1,F2,F123
F1,0.532,-0.125,0.483
F2,-0.125,0.967,0.129
F123,0.483,0.129,0.501


Q_3


,F1,F2,F123
F1,1.0,0.0,0.0
F2,0.0,1.0,0.0
F123,0.0,0.0,1.0


## 6. Structural Variance Information (SVI)

`pca_tools` calculates the same cumulative diagonal values as `self_explanatory_power` and also reports variable-level $R^2$. Here the curves answer a more useful question than a loading alone: **which variables are structurally represented by the PCA subspace as each component is added?**

The final component must bring both measures to one in this full-rank example. Differences in the earlier components distinguish common structure from factor-specific structure.

In [8]:
svi_summary = pd.concat(
    {
        "self_explanatory_power": model.svi_["self_explanatory_power"],
        "variable_R2": model.svi_["R2"],
    },
    axis=1,
)
svi_summary.round(3)

self_explanatory_power             variable_R2            
                        PC1    PC2  PC3         PC1    PC2  PC3
F1                    0.470  0.532  1.0       0.921  0.985  1.0
F2                    0.029  0.967  1.0       0.058  0.999  1.0
F123                  0.501  0.501  1.0       0.983  0.983  1.0

In [9]:
model.svi_plot("F1")

alt.LayerChart(...)

In [10]:
model.svi_plot("F123")

alt.LayerChart(...)

## Takeaway

This pipe-mixing system shows why SVI is complementary to a loading plot. A loading describes one component at a time, while $Q_A$, self-explanatory power, and variable $R^2$ describe the cumulative PCA subspace. In particular, they distinguish the strong shared structure between $F_1$ and the outlet flow $F_{123}$ from the weaker contribution of $F_2$ and the unmeasured $F_3$ contribution.

Reference: Camacho, J., Picó, J. and Ferrer, A. (2010). *Data understanding with PCA: Structural and Variance Information plots*. Chemometrics and Intelligent Laboratory Systems, 100(1), 48–56. https://doi.org/10.1016/j.chemolab.2009.10.005